## Downloading the datasets

Downloads the six datasets into `data/raw/`. The first four (toddler, child, adolescent, adult) were collected by F. Thabtah through the ASDTests app and are the main datasets. `toddler_saudi` and `adult_kaggle` come from other sources and are only used for the cross-source test (RQ2).

### Kaggle access

Three datasets are hosted on Kaggle, which needs an API token:

1. Kaggle → *Settings* → *API* → **Create New Token**
2. Move the downloaded `kaggle.json` to `~/.kaggle/kaggle.json`
3. Run `chmod 600 ~/.kaggle/kaggle.json`

The UCI datasets download without an account.

In [1]:
import io
import zipfile
from pathlib import Path

import requests

RAW = Path("../data/raw")

UCI = {
    "child": "419/autistic+spectrum+disorder+screening+data+for+children",
    "adolescent": "420/autistic+spectrum+disorder+screening+data+for+adolescent",
    "adult": "426/autism+screening+adult",
}
KAGGLE = {
    "toddler": "fabdelja/autism-screening-for-toddlers",
    "toddler_saudi": "asdpredictioninsaudi/asd-screening-data-for-toddlers-in-saudi-arabia",
    "adult_kaggle": "shivamshinde123/autismprediction",
}

### UCI datasets

The UCI files come as a zip containing an `.arff` file and a Word document describing the data.

In [2]:
for name, ref in UCI.items():
    out = RAW / name
    out.mkdir(parents=True, exist_ok=True)
    r = requests.get(f"https://archive.ics.uci.edu/static/public/{ref}.zip", timeout=60)
    r.raise_for_status()
    zipfile.ZipFile(io.BytesIO(r.content)).extractall(out)
    print("downloaded", name)

downloaded child


downloaded adolescent


downloaded adult


### Kaggle datasets

In [3]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()

for name, ref in KAGGLE.items():
    api.dataset_download_files(ref, path=RAW / name, unzip=True)
    print("downloaded", name)

Dataset URL: https://www.kaggle.com/datasets/fabdelja/autism-screening-for-toddlers


downloaded toddler
Dataset URL: https://www.kaggle.com/datasets/asdpredictioninsaudi/asd-screening-data-for-toddlers-in-saudi-arabia


downloaded toddler_saudi
Dataset URL: https://www.kaggle.com/datasets/shivamshinde123/autismprediction


downloaded adult_kaggle


### What was downloaded

In [4]:
for folder in sorted(RAW.iterdir()):
    if folder.is_dir():
        print(folder.name)
        for f in sorted(folder.iterdir()):
            print(f"    {f.name:<80} {f.stat().st_size / 1024:6.1f} KB")

adolescent
    Autism-Adolescent-Data.arff                                                         9.7 KB
    Autism-Screening-Adolescent-Data Description.docx                                  37.9 KB
adult
    Autism-Adult-Data.arff                                                             58.4 KB
    Autism-Screening-Adult-Data Description.docx                                       38.5 KB
adult_kaggle
    test.csv                                                                           19.0 KB
    train.csv                                                                          77.9 KB
child
    Autism-Child-Data.arff                                                             24.7 KB
    Autism-Screening-Child-Data Description.docx                                       37.9 KB
toddler
    Autism_Screening_Data_Combined.csv                                                209.7 KB
    Toddler Autism dataset July 2018.csv                                               68.5 KB
    To

Each folder also contains the description document written by the dataset authors (`.docx` or `.pdf`). These were used to write notebook 02.

`adult_kaggle` contains a `test.csv` without labels (it came from a Kaggle competition), so only `train.csv` is used.

**Licences.** The UCI datasets are released under CC BY 4.0. The Kaggle pages do not state a licence, so those datasets are used for research only and credited to their authors.